确认自己的GPU

In [15]:
!nvidia-smi

Fri Sep  4 17:48:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.71                 KMD Version: 610.71        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   53C    P8              5W /   61W |     114MiB /   8151MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

我们与需要表达我们的意见：使用GPU来计算

In [16]:
import torch
from torch import nn
torch.device('cpu'),torch.cuda.device('cuda')

(device(type='cpu'), <torch.cuda.device at 0x1ac017b6a10>)

In [17]:
torch.cuda.device_count()

1

这两个函数允许我们在请求的GPU不存在的情况下运行代码

In [18]:
def try_gpu(i=0):
    '''如果存在，返回gpu(i)，否则返回cpu()'''
    if torch.cuda.device_count() >= i+1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')
def try_all_gpus():
    devices=[
        torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count())
    ]
    return devices if devices else [torch.device('cpu')]
try_gpu(),try_gpu(10),try_all_gpus()

(device(type='cuda', index=0),
 device(type='cpu'),
 [device(type='cuda', index=0)])

查询张量所在的设备

In [19]:
x=torch.tensor([1,2,3])
x.device

device(type='cpu')

可以在创建的时候放入GPU上

In [20]:
x=torch.tensor([1.0,2.0,3.0],device='cuda')
x

tensor([1., 2., 3.], device='cuda:0')

In [21]:
Y=torch.tensor([1,2,3],device='cuda')
Y

tensor([1, 2, 3], device='cuda:0')

In [22]:
Z=x.cuda(0)
print(x)
print(Z)

tensor([1., 2., 3.], device='cuda:0')
tensor([1., 2., 3.], device='cuda:0')


In [23]:
Z+x

tensor([2., 4., 6.], device='cuda:0')

神经网络与GPU

In [24]:
net=nn.Sequential(nn.Linear(3,1))
net=net.to(device=try_gpu())
net(x)

tensor([-2.5794], device='cuda:0', grad_fn=<ViewBackward0>)

确认模型参数存储在同一个GPU上

In [25]:
net[0].weight.data.device

device(type='cuda', index=0)